**Intelligent Vision Systems · Lab 6 · Camera Calibration**
Companion to **Lecture 6 — Camera Calibration**
Dr. Muhammad Kazim · Department of Intelligent Systems, University of Lahore

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Kazimbalti/intelligent-vision-systems/blob/main/LAB/Lab_6/09-pi-camera-in-15-minutes.ipynb) &nbsp; [Lecture 6 slides](https://kazimbalti.github.io/intelligent-vision-systems/lectures/06_Camera_Calibration.html#/watch-and-learn) &nbsp; [All Lab 6 notebooks](https://kazimbalti.github.io/intelligent-vision-systems/labs.html#lab-6-calibration)

# Lab 6I · Calibrate a Raspberry Pi Camera in 15 Minutes

_Lecture 6, video lab · The Drone Dojo._

> 🎬 **Watch first:** [Calibrate OpenCV Camera In Under 15 Minutes](https://www.youtube.com/watch?v=O-7TkOKeK2M) (The Drone Dojo).
> Step-by-step notes for this video: [Tutorial 08 · Raspberry Pi in 15 minutes](https://kazimbalti.github.io/intelligent-vision-systems/tutorials/08_camera_calibration_videos.html#pi-in-15-minutes).

<a href="https://www.youtube.com/watch?v=O-7TkOKeK2M"><img src="https://img.youtube.com/vi/O-7TkOKeK2M/mqdefault.jpg" width="320" alt="Calibrate OpenCV Camera In Under 15 Minutes"></a>

## What the video does, in plain words

1. **Print** a chessboard on A4, glue it to something flat, and **measure one square** (26 mm in the video).
2. Run a script **on the Raspberry Pi** at **640 × 480**. It watches the camera and **keeps 20 good pictures** of the board
   by itself: you just move the board around (show the **edges** of the image first, then all positions).
3. The script **calibrates** and saves the **camera matrix** and **distortion coefficients**.
4. **Test it like an engineer:** put an **ArUco marker** in front of the camera, read its position (x, y, z), slide it
   **6 cm** with a ruler and check that the program also says **≈ 6 cm**. If not, recalibrate.

This notebook does every step twice: first on a **virtual Pi camera** (so you can run it anywhere, even on Colab, and we
know the right answer), then with **your real Pi camera** using the two scripts `pi_calibrate_20.py` and `aruco_pose_test.py`.

In [ ]:
# Setup (only needed on Google Colab or a fresh environment).
# Uncomment the next line to install the packages this notebook uses:
# !pip install -q opencv-python numpy matplotlib

import os, urllib.request

# Fetch the lab files if they are not next to the notebook.
RAW = "https://raw.githubusercontent.com/Kazimbalti/intelligent-vision-systems/main/LAB/Lab_6"
for f in ['lab6_checks.py', 'pi_calibrate_20.py', 'aruco_pose_test.py']:
    if not os.path.exists(f):
        os.makedirs(os.path.dirname(f) or ".", exist_ok=True)
        urllib.request.urlretrieve(f"{RAW}/{f}", f)
print("data ready")

In [ ]:
import glob, json, os, pickle, time
import numpy as np
import matplotlib.pyplot as plt
import cv2
import lab6_checks as C
np.set_printoptions(precision=4, suppress=True)
print("OpenCV", cv2.__version__)

## Part A — on the Raspberry Pi (do this in the lab)

Our scripts follow the video but are updated for **Raspberry Pi OS Bookworm** and **OpenCV 5**:

| In the video | What we do today | Why |
|---|---|---|
| `sudo pip install opencv-python numpy` | `sudo apt install python3-opencv python3-picamera2` (or a `venv --system-site-packages`) | `sudo pip` breaks the system Python on Bookworm |
| webcam stream (`cv2.VideoCapture`) for the Pi camera | **Picamera2** (`--usb N` for a USB webcam) | Pi cameras use libcamera; `VideoCapture(0)` usually fails |
| `cv2.aruco.estimatePoseSingleMarkers` | `cv2.solvePnP(..., SOLVEPNP_IPPE_SQUARE)` | removed in OpenCV ≥ 4.7 |

```bash
# 1. log in (SSH for the terminal, VNC for the camera window)
ssh pi@raspberrypi.local
rpicam-hello --list-cameras          # is the camera detected?

# 2. get the scripts and the chessboard
mkdir -p ~/calib && cd ~/calib
for f in pi_calibrate_20.py aruco_pose_test.py; do
  wget https://raw.githubusercontent.com/Kazimbalti/intelligent-vision-systems/main/LAB/Lab_6/$f; done
wget https://raw.githubusercontent.com/opencv/opencv/4.x/doc/pattern.png      # print on A4 (9 x 6 inner corners)
sudo apt install -y python3-opencv python3-picamera2

# 3. calibrate (run from a VNC desktop terminal so the window can open)
python3 pi_calibrate_20.py --mm 26 --width 640 --height 480

# 4. make, print and measure an ArUco marker, then test with a ruler
python3 aruco_pose_test.py --make-marker --id 72
python3 aruco_pose_test.py --size 200 --id 72 --width 640 --height 480
```

> **Same resolution everywhere.** K is only valid for the size you calibrated at. If you calibrate at 640 × 480, run the
> marker test (and your project) at 640 × 480 too.

## Part B — the same workflow on a virtual Pi camera

We pretend to be a Pi camera at 640 × 480 with a known lens, render a **video** of you moving the board around, and run the
**exact frame filter** from `pi_calibrate_20.py`.

In [ ]:
from pi_calibrate_20 import board_signature, is_new     # the functions the Pi script uses

W, H = 640, 480
K_true = np.array([[505.0, 0, 322.0], [0, 503.0, 238.0], [0, 0, 1]])   # a Pi Camera v2-like lens at 640x480
D_true = np.array([0.18, -0.42, 0.001, -0.0007, 0.25])
SQ = 26.0                                                               # mm, as in the video

# the printed pattern: 10 x 7 squares -> 9 x 6 inner corners; 0.2 mm per texture pixel (130 px per square)
s = int(SQ / 0.2)
board = np.full((7 * s + 2 * s, 10 * s + 2 * s), 255, np.uint8)
for r in range(7):
    for c in range(10):
        if (r + c) % 2 == 0: board[s + r * s:s + (r + 1) * s, s + c * s:s + (c + 1) * s] = 0

poses = C.random_poses(140, seed=7, dist_mm=(330, 700), tilt_deg=30, board_mm=(board.shape[1] * 0.2, board.shape[0] * 0.2))
frames = [C.render_board(board, 0.2, K_true, D_true, r, t, (W, H), background=170) for r, t in poses]
print(len(frames), "video frames rendered")
fig, ax = plt.subplots(1, 5, figsize=(22, 3.5))
for a, f in zip(ax, frames[::28]): a.imshow(f, cmap="gray"); a.axis("off")
plt.show()

### Keep only good, *different* frames

A frame is **good** if the whole board is found. It is **different** if its (x, y, size, skew) signature is far enough
from every frame kept so far. Complete the loop: stop at 20 frames.

In [ ]:
pattern, novelty, n_target = (9, 6), 0.15, 20
objp = C.board_points(*pattern, SQ)
kept, objpoints, imgpoints, kept_idx = [], [], [], []
for i, f in enumerate(frames):
    ok, corners = C.find_corners(f, pattern)
    if not ok:
        continue
    sig = board_signature(corners, W, H)
    # TODO: if this board is new (is_new) keep it: append sig, objp, corners and i; stop at n_target frames
    pass

print(len(kept), "good frames kept, from frame indices", kept_idx)

In [ ]:
C.check_good_frame_filter(kept, n_target)

In [ ]:
rms, K, D, rvecs, tvecs = cv2.calibrateCamera(objpoints, imgpoints, (W, H), None, None)
print(f"RMS {rms:.3f} px"); print("K =\n", K); print("K_true =\n", K_true)
print("fx error: %.2f %%" % (100 * abs(K[0, 0] - K_true[0, 0]) / K_true[0, 0]))
np.savetxt("cameraMatrix.txt", K, delimiter=","); np.savetxt("cameraDistortion.txt", D, delimiter=",")

## Part C — the ruler test with an ArUco marker

The video's test: centre the marker (x ≈ 0), slide it **6 cm** along x, and read the new x. We render a **200 mm** marker
(ID 72, `DICT_ARUCO_ORIGINAL`, like the video) at x = 0 and at x = +60 mm, then measure both positions with `solvePnP`
and **your** calibration.

In [ ]:
dictionary = cv2.aruco.getPredefinedDictionary(cv2.aruco.DICT_ARUCO_ORIGINAL)
marker = cv2.copyMakeBorder(cv2.aruco.generateImageMarker(dictionary, 72, 800), 100, 100, 100, 100, cv2.BORDER_CONSTANT, value=255)
MARKER_MM = 200.0                     # the black square is 800 px -> 0.25 mm per pixel
mm_px = MARKER_MM / 800
centre = 500 * mm_px                  # marker centre in the texture, mm

def shot(shift_mm):
    return C.render_board(marker, mm_px, K_true, D_true, [0.05, -0.08, 0.0], [-centre + shift_mm, -centre, 900], (W, H))

def marker_position(img, K, D, size=MARKER_MM):
    detector = cv2.aruco.ArucoDetector(dictionary, cv2.aruco.DetectorParameters())
    corners, ids, _ = detector.detectMarkers(img)
    if ids is None: return None
    s = size / 2
    obj = np.array([[-s, s, 0], [s, s, 0], [s, -s, 0], [-s, -s, 0]], np.float32)
    ok, rvec, tvec = cv2.solvePnP(obj, corners[0].reshape(4, 2), K, D, flags=cv2.SOLVEPNP_IPPE_SQUARE)
    return tvec.ravel()

a, b = shot(0), shot(60)
pa, pb = marker_position(a, K, D), marker_position(b, K, D)
print("before:", pa.round(1), "mm\nafter: ", pb.round(1), "mm")
fig, ax = plt.subplots(1, 2, figsize=(14, 5)); ax[0].imshow(a, cmap="gray"); ax[1].imshow(b, cmap="gray")
for x in ax: x.axis("off")
plt.show()

**Quiz:** compute how far the marker moved along x, in mm, from `pa` and `pb`.

In [ ]:
shift_mm = np.nan   # TODO
print(f"measured shift: {shift_mm:.1f} mm")

In [ ]:
C.check_marker_shift(shift_mm, 60)

**What if the calibration is wrong?** Try the same test with a "datasheet guess" for K and **no** distortion:

In [ ]:
K_guess = np.array([[480.0, 0, 320], [0, 480.0, 240], [0, 0, 1]])
pa_g, pb_g = marker_position(a, K_guess, np.zeros(5)), marker_position(b, K_guess, np.zeros(5))
print("uncalibrated: shift = %.1f mm, distance z = %.0f mm (true 900)" % (pb_g[0] - pa_g[0], pa_g[2]))
print("calibrated:   shift = %.1f mm, distance z = %.0f mm" % (pb[0] - pa[0], pa[2]))

## Your turn

1. On the **real Pi**: calibrate with `pi_calibrate_20.py`, then do the 6 cm ruler test with `aruco_pose_test.py`. Report
   the RMS, K, the measured shift and the distance error. Take a phone photo of your setup.
2. Run the ruler test **at 1280 × 720** with the 640 × 480 calibration. What happens, and how would you fix K without
   recalibrating? (Hint: Lecture 6, "scaling the image scales K".)
3. In Part B set `novelty = 0.02`. How many of the 20 kept frames are now nearly the same, and how does the fx error change?
   Why does the video say "show all the edges first"?